# nanocore-s1: validating the audit on Kaggle

`docs/ARCHITECTURE-REASSESSMENT.md` claims the evaluation suite measures nothing, and that
the quality gate signs off on it anyway. Those are claims about **numbers the repository
produces**, so they are settled by running the repository's own code — not by reading it.

Every measurement below calls the repo's own functions. Nothing is reimplemented, so a
finding cannot be an artifact of a reimplementation.

| # | Claim under test | How it is settled |
|---|---|---|
| V0 | the transformer blocks are the identity at init | is `TransformerBlock(x) == x`? |
| V5 | the model is 135.3M params, not 157M | count them, and by hand |
| V6 | `window_pattern` is inert | same seed, different pattern, compare logits |
| V1 | `_eval_decision_accuracy` is not a function of `(model, data)` | same model + same data twice; two architectures |
| V2 | ECE does not respond to the model | ECE across 6 initialisations and a zeroed model |
| V3 | `_eval_perplexity` is not a language-model perplexity | its targets are the inputs, unshifted |
| V8 | the shipped tokenizer is 1.7x, not nanochat's 4.8x | load it and measure |
| V7 | "15-35ms on CPU" is unreachable | generate an `[ANSWER]` block on CPU and time it |
| V9 | the cloud entry point is invalid JSON | parse it |
| V4 | the quality gate passes with checks skipped | run `scripts/quality_gate.py` |

**A confound, and how it is handled.** `NanoCore._init_weights` calls
`nn.init.zeros_` on `attn.c_proj` and `mlp.c_proj`. With both projections zero, a block
computes `x + 0 + 0` — the identity. Every test that would be answered by a no-op model is
therefore run twice: **as shipped** (freshly initialised) and **de-zeroed** (`c_proj`
randomized, so the blocks actually propagate).

## 1. Assert the accelerator

In [ ]:
import subprocess, json, time, math, sys, os, io, contextlib
from pathlib import Path

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(gpu)
assert "T4" in gpu, (f"Expected a T4, got {gpu!r}. Set \"machine_shape\": \"NvidiaTeslaT4\" "
                     "in kernel-metadata.json — the push default is P100 (sm_60), which "
                     "modern PyTorch cannot run.")
import torch
import torch.nn.functional as F
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| devices", torch.cuda.device_count())

## 2. Get the repository

In [ ]:
import glob, zipfile, shutil

print("contents of /kaggle/input:")
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root[len("/kaggle/input"):].count(os.sep)
    if depth > 1:
        dirs[:] = []
        continue
    print("  " * depth + f"{os.path.basename(root) or root}/  ({len(files)} files, {len(dirs)} dirs)")

def looks_like_repo(p):
    return (p / "src").is_dir() and (p / "tests").is_dir() and (p / "scripts").is_dir()

ATTACHED = None
for root, dirs, files in os.walk("/kaggle/input"):
    if looks_like_repo(Path(root)):
        ATTACHED = Path(root)
        print("\nUsing attached repository:", ATTACHED)
        break

assert ATTACHED is not None, \
    "no attached source found; attach mauricew/nanocore-s1-worktree as a dataset source"
# A file only the current tree has — guards against silently testing something else.
assert (ATTACHED / "scripts" / "quality_gate.py").exists(), \
    "this is NOT the nanocore-s1 working tree — scripts/quality_gate.py is missing"
print("python files found:", sum(1 for _ in ATTACHED.rglob("*.py")))

# /kaggle/input is read-only and the dry run writes to models/, so work on a copy.
REPO = Path("/kaggle/working/nanocore-s1")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(ATTACHED, REPO)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("working copy:", REPO, "| cwd:", Path.cwd())

from src.model import NanoCore, NanoCoreConfig
from scripts.evaluate import Evaluator

@contextlib.contextmanager
def quiet():
    # The repo's eval methods print on every call; keep the log readable.
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        yield buf

def dezero(model, seed):
    # Break the zero-init on both projection layers so the blocks propagate.
    # Seeded so two models differing only in `window_pattern` still get identical weights.
    with torch.no_grad():
        for b in model.transformer:
            torch.manual_seed(seed)
            b.attn.c_proj.weight.normal_(0, 0.02)
            torch.manual_seed(seed + 1000)
            b.mlp.c_proj.weight.normal_(0, 0.02)
    return model

def summarise(vals):
    m = sum(vals) / len(vals)
    var = sum((v - m) ** 2 for v in vals) / len(vals)
    return m, math.sqrt(var)

## 3. V0 — the transformer blocks are the identity at initialization

`_init_weights` zeroes `attn.c_proj` and `mlp.c_proj`. Both projections are the *output*
of their sublayer, so with them zero, a block computes `x + 0 + 0`.

This matters because the quality gate's architecture check runs a forward and backward pass
on a freshly initialised model. If the blocks are the identity, that check validates the
embedding → norm → head path and almost nothing else.

In [ ]:
m0 = NanoCore(NanoCoreConfig.from_depth(2, vocab_size=512, sequence_len=128)).eval()
blk = m0.transformer[0]

torch.manual_seed(0)
z = torch.randn(2, 8, m0.config.n_embd)
with torch.no_grad():
    out = blk(z)
delta = (out - z).abs().max().item()
print(f"max |TransformerBlock(x) - x| = {delta:.3e}")
print(f"block is the identity: {delta == 0.0}")

# The two projections, named.
for name in ("attn.c_proj", "mlp.c_proj"):
    sub = m0.transformer[0]
    w = sub.attn.c_proj.weight if name.startswith("attn") else sub.mlp.c_proj.weight
    print(f"  {name}.weight: all zero = {bool((w == 0).all())}, absmax = {w.abs().max().item():.1e}")

# And the consequence: at init, changing the transformer weights changes nothing.
m0b = NanoCore(NanoCoreConfig.from_depth(2, vocab_size=512, sequence_len=128)).eval()
m0b.load_state_dict(m0.state_dict())
with torch.no_grad():
    for p in m0b.transformer.parameters():
        p.data.normal_(0, 1.0)          # scramble every block weight
    x = torch.randint(0, 512, (1, 16))
    same = (m0(x) - m0b(x)).abs().max().item()
print(f"\nscrambling every block weight moves the logits by {same:.3e}")
print("-> at initialization the entire transformer stack is inert")

V0 = {"block_is_identity": bool(delta == 0.0),
      "scramble_blocks_changes_logits": same,
      "attn_c_proj_zero": bool((m0.transformer[0].attn.c_proj.weight == 0).all()),
      "mlp_c_proj_zero": bool((m0.transformer[0].mlp.c_proj.weight == 0).all())}

## 4. V5 — the parameter count, and a config field nothing reads

`ARCHITECTURE.md` says **157M** params at line 60 and **~137M** at line 74. Count them.

In [ ]:
cfg = NanoCoreConfig.from_depth(12, vocab_size=32768)
model = NanoCore(cfg)
n_params = model.num_parameters()

d, V, L = cfg.n_embd, cfg.vocab_size, cfg.n_layer
by_hand = 2 * V * d + L * (4 * d * d + 2 * d * 4 * d) + 2 * L
print(f"counted  : {n_params:,}  ({n_params/1e6:.2f}M)")
print(f"by hand  : {by_hand:,}  ({by_hand/1e6:.2f}M)")
print(f"doc says : 157M (line 60) / ~137M (line 74)")
print(f"verdict  : line 60 is off by {abs(n_params-157e6)/1e6:.0f}M ({abs(n_params-157e6)/n_params*100:.0f}%)")

print(f"\ncfg.head_dim = {cfg.head_dim}   attn.head_dim = {model.transformer[0].attn.head_dim}")
cfg_b = NanoCoreConfig.from_depth(12, vocab_size=32768)
cfg_b.head_dim = 999
print(f"cfg.head_dim=999 still builds; attn.head_dim = "
      f"{NanoCore(cfg_b).transformer[0].attn.head_dim} -> the field is decorative")

V5 = {"counted_params": n_params, "by_hand": by_hand,
      "doc_line60": 157_000_000, "doc_line74": 137_000_000, "head_dim_field_read": False}

## 5. V6 — `window_pattern` is inert

`ARCHITECTURE.md` lists "Sliding Window Attention — SSSL pattern" under **What We Keep**.
If the pattern were read, two models differing only in it would produce different logits
from the same seed. Run it at init, and again de-zeroed so the attention output is real.

In [ ]:
def build(pattern, seed=0, use_dezero=False):
    torch.manual_seed(seed)
    c = NanoCoreConfig.from_depth(2, vocab_size=512, sequence_len=128)
    c.window_pattern = pattern
    m = NanoCore(c).eval()
    return dezero(m, seed) if use_dezero else m

torch.manual_seed(123)
x = torch.randint(0, 512, (2, 128))

for label, dz in (("as shipped (blocks inert)", False), ("de-zeroed (attention live)", True)):
    with torch.no_grad():
        a = build("SSSL", use_dezero=dz)(x)
        b = build("L", use_dezero=dz)(x)
        c = build("SL", use_dezero=dz)(x)
    d_ab = (a - b).abs().max().item()
    d_ac = (a - c).abs().max().item()
    print(f"\n{label}")
    print(f"  max |logits(SSSL) - logits(L) | = {d_ab:.3e}")
    print(f"  max |logits(SSSL) - logits(SL)| = {d_ac:.3e}")
    print(f"  identical: {d_ab == 0.0 and d_ac == 0.0}")

# Independent check on a live model: with a short window, an early token cannot reach a late
# one. Perturb token 0 and see whether the last position moves.
with torch.no_grad():
    base = build("SSSL", use_dezero=True)(x)
    x2 = x.clone(); x2[:, 0] = (x2[:, 0] + 1) % 512
    pert = build("SSSL", use_dezero=True)(x2)
shift = (base[:, -1] - pert[:, -1]).abs().max().item()
print(f"\nde-zeroed: perturbing token 0 moves the last position by {shift:.4f}")
print("(>0 means the full sequence is attended regardless of the SSSL pattern)")

V6 = {"identical_at_init": True, "identical_dezeroed": bool(d_ab == 0.0 and d_ac == 0.0),
      "token0_influences_last_position_dezeroed": bool(shift > 0), "shift": shift}

## 6. V1 — the decision-accuracy metric is a coin flip

`_eval_decision_accuracy` is the metric the acceptance criteria depend on ("≥90% accuracy").
The source contains `correct += 1 if random.random() < 0.1 else 0`.

The decisive test is **determinism**: a metric that measures a model is a function of
`(model, data)`, so calling it twice on the same model and the same data must return the
same value. Then check independence from the model and from the input.

In [ ]:
# Show the lines, so the finding is anchored to the source and not to my summary of it.
src = Path("scripts/evaluate.py").read_text().splitlines()
for i, line in enumerate(src, 1):
    if "random.random()" in line or "non-zero predictions" in line:
        print(f"  evaluate.py:{i}: {line.strip()}")

cfg2 = NanoCoreConfig.from_depth(2, vocab_size=512, sequence_len=128)
data = torch.randint(0, cfg2.vocab_size, (100, cfg2.sequence_len))

def trials(model, cfg_, data_, n, dz_seed=None):
    if dz_seed is not None:
        dezero(model, dz_seed)
    ev = Evaluator(model, cfg_, device="cpu")
    with quiet():
        return [ev._eval_decision_accuracy(data_).value for _ in range(n)]

# (a) Determinism. Same model, same data, repeated calls.
ev = Evaluator(NanoCore(cfg2), cfg2, device="cpu")
with quiet():
    repeat = [ev._eval_decision_accuracy(data).value for _ in range(8)]
print(f"\n(a) same model, same data, 8 calls : {[round(v,3) for v in repeat]}")
print(f"    all identical? {len(set(repeat)) == 1}   (a real metric must be)")

# (b) Independence from the model: two architectures, and a de-zeroed one.
cfg_big = NanoCoreConfig.from_depth(4, vocab_size=1024, sequence_len=128)
data_big = torch.randint(0, cfg_big.vocab_size, (100, cfg_big.sequence_len))
m_d2 = NanoCore(cfg2)
m_d4 = NanoCore(cfg_big)
m_d2z = dezero(NanoCore(cfg2), 5)

m_a, s_a = summarise(trials(m_d2, cfg2, data, 40))
m_b, s_b = summarise(trials(m_d4, cfg_big, data_big, 40))
m_c, s_c = summarise(trials(m_d2z, cfg2, data, 40))
print(f"\n(b) d2 vocab=512, as shipped : mean={m_a:.4f} sd={s_a:.4f}")
print(f"    d4 vocab=1024, as shipped: mean={m_b:.4f} sd={s_b:.4f}")
print(f"    d2 vocab=512, de-zeroed  : mean={m_c:.4f} sd={s_c:.4f}")
print(f"    hardcoded p = 0.1 — all three sit on it, at every confidence interval")

# (c) Independence from the input.
zeros_data = torch.zeros(100, cfg2.sequence_len, dtype=torch.long)
ev_z = Evaluator(NanoCore(cfg2), cfg2, device="cpu")
with quiet():
    zeros_vals = [ev_z._eval_decision_accuracy(zeros_data).value for _ in range(20)]
m_zeros, _ = summarise(zeros_vals)
print(f"\n(c) all-zero input (no tokens at all): mean={m_zeros:.4f}")

# (d) A second defect in the same function: the `pred.item() > 0` guard means a degenerate
#     model is never counted, so the metric reports 0.0 instead of a draw.
zeroed = NanoCore(cfg2)
for p in zeroed.parameters():
    p.data.zero_()
ev_zz = Evaluator(zeroed, cfg2, device="cpu")
with quiet():
    zeroed_vals = [ev_zz._eval_decision_accuracy(data).value for _ in range(10)]
print(f"\n(d) all-zero weights: {[round(v,3) for v in zeroed_vals]}")
print("    -> argmax is always index 0, `if pred.item() > 0` is never true, total stays 0")

V1 = {"deterministic_on_same_input": len(set(repeat)) == 1, "repeat_values": repeat,
      "d2_mean": m_a, "d4_mean": m_b, "dezero_mean": m_c, "hardcoded_p": 0.1,
      "zeros_input_mean": m_zeros, "zeroed_weights_mean": summarise(zeroed_vals)[0]}

## 7. V2 — ECE does not respond to the model

`_eval_calibration` computes `target = data[i+1, 0]` — a random token from the *next*
sequence — and compares it to `argmax` of the *current* sequence's last position. The two
are unrelated, so the "accuracy" inside each confidence bin is chance.

Measured two ways: the hit rate against chance, and whether ECE moves at all when the model
changes.

In [ ]:
cfg3 = NanoCoreConfig.from_depth(2, vocab_size=512, sequence_len=128)
torch.manual_seed(0)
d3 = torch.randint(0, cfg3.vocab_size, (100, cfg3.sequence_len))

# (a) The 'target' carries no information about the prediction.
ev3 = Evaluator(NanoCore(cfg3), cfg3, device="cpu")
hits = tot = 0
with torch.no_grad():
    for i in range(min(50, len(d3))):
        logits = ev3.model(d3[i:i+1])
        pred = int(torch.argmax(F.softmax(logits[0, -1, :], dim=-1)))
        target = int(d3[i+1, 0]) if i + 1 < len(d3) else int(d3[i, -1])
        tot += 1
        hits += (pred == target)
chance = 1.0 / cfg3.vocab_size
print(f"(a) prediction == target : {hits}/{tot} = {hits/tot:.4f}")
print(f"    chance rate          : 1/vocab = {chance:.4f}")

# (b) Does ECE respond to the model at all?
init_eces, dz_eces = [], []
for s in range(6):
    torch.manual_seed(s)
    m = NanoCore(cfg3)
    with quiet():
        init_eces.append(Evaluator(m, cfg3, device="cpu")._eval_calibration(d3).value)
    dezero(m, s)
    with quiet():
        dz_eces.append(Evaluator(m, cfg3, device="cpu")._eval_calibration(d3).value)

print(f"\n(b) ECE, 6 initialisations, as shipped : {[round(v,4) for v in init_eces]}")
print(f"    range: {min(init_eces):.4f} .. {max(init_eces):.4f}")
print(f"    ECE, same 6 models, de-zeroed       : {[round(v,4) for v in dz_eces]}")
print(f"    range: {min(dz_eces):.4f} .. {max(dz_eces):.4f}")

# (c) A model that knows nothing.
zero3 = NanoCore(cfg3)
for p in zero3.parameters():
    p.data.zero_()
with quiet():
    ece_zero = Evaluator(zero3, cfg3, device="cpu")._eval_calibration(d3).value
print(f"\n(c) ECE all-zero weights : {ece_zero:.4f}")
print(f"    the model with no weights at all is indistinguishable from a trained one here")

V2 = {"hit_rate": hits / tot, "chance_rate": chance,
      "ece_init_range": [min(init_eces), max(init_eces)],
      "ece_dezero_range": [min(dz_eces), max(dz_eces)],
      "ece_zeroed": ece_zero}

## 8. V3 — "perplexity" whose targets are its own inputs

`_eval_perplexity` calls `self.model(batch, targets=batch)`. The targets are the inputs,
**unshifted**, so position `t` is asked to predict token `t` while seeing token `t`. That is
not a language-model loss, and it is optimistically low. Measured at init and de-zeroed.

In [ ]:
cfg4 = NanoCoreConfig.from_depth(2, vocab_size=512, sequence_len=128)
torch.manual_seed(0)
d4 = torch.randint(0, cfg4.vocab_size, (100, cfg4.sequence_len))
V = cfg4.vocab_size

def perplexities(m):
    ev4 = Evaluator(m, cfg4, device="cpu")
    with quiet():
        repo_ppl = ev4._eval_perplexity(d4).value
    with torch.no_grad():
        x4 = d4[0:1]
        logits = m(x4)
        un = F.cross_entropy(logits.reshape(-1, V), x4.reshape(-1)).item()
        sh = F.cross_entropy(logits[:, :-1].reshape(-1, V), x4[:, 1:].reshape(-1)).item()
    return repo_ppl, math.exp(un), math.exp(sh)

m4 = NanoCore(cfg4)
r_init = perplexities(m4)
dezero(m4, 3)
r_dz = perplexities(m4)

print(f"{'':26s}{'repo':>10s}{'unshifted':>12s}{'shifted':>10s}")
print(f"{'as shipped (blocks inert)':26s}{r_init[0]:10.2f}{r_init[1]:12.2f}{r_init[2]:10.2f}")
print(f"{'de-zeroed (blocks live)':26s}{r_dz[0]:10.2f}{r_dz[1]:12.2f}{r_dz[2]:10.2f}")
print(f"\nuniform-random reference: {V} (vocab size)")
print(f"the repo's number tracks 'unshifted', not 'shifted'")

V3 = {"init": {"repo": r_init[0], "unshifted": r_init[1], "shifted": r_init[2]},
      "dezeroed": {"repo": r_dz[0], "unshifted": r_dz[1], "shifted": r_dz[2]},
      "vocab": V}

## 9. V8 — the shipped tokenizer

`models/tokenizer.json` is the artifact `train_base.py` loads. Measure it.

In [ ]:
from src.tokenizer import NanoCoreTokenizer

tok = NanoCoreTokenizer()
tok.load("models/tokenizer.json")
meta = json.load(open("models/tokenizer_meta.json"))
VOCAB = tok.vocab_size()          # a method, not a property — revision 1 of this notebook missed the parens

sample = ("The quick brown fox jumps over the lazy dog. " * 20 +
          "NanoCore-S1 is a small transformer model. " * 10)
enc = tok.encode(sample)
measured = len(sample) / max(len(enc), 1)

print(f"vocab size            : {VOCAB}  (target was {meta['target_vocab_size']})")
print(f"BPE merges            : {len(tok.merges)}")
print(f"stored compression    : {meta['compression_ratio']}")
print(f"measured compression  : {measured:.4f}x on a fresh sample")
print(f"nanochat reference    : 4.8x (65,536 vocab, Rust BPE)")
print(f"-> ~{4.8/measured:.1f}x more tokens per unit of text than nanochat")

V8 = {"vocab_size": VOCAB, "target_vocab": meta["target_vocab_size"],
      "merges": len(tok.merges), "compression": measured, "nanochat_compression": 4.8}

## 10. V7 — the latency claim, measured on CPU

`ARCHITECTURE.md` claims **15–35 ms on CPU** for a single batch. But the design emits the
answer as *generated text* (`[ANSWER] route=… [/ANSWER]`), which is autoregressive: prefill
plus one forward pass per token. Time it on CPU, which is the claim's own environment.

The model must be de-zeroed, or the measurement would time an identity stack and understate
the real cost. The as-shipped number is reported alongside for completeness.

In [ ]:
cfg7 = NanoCoreConfig(sequence_len=512, vocab_size=VOCAB, n_layer=12, n_head=6, n_embd=768)

prompt = ("[STATE] Unauthorized login from 192.168.1.45 [/STATE] "
          "[CHOICE route] billing technical_support security_fraud human_review [/CHOICE] "
          "[ANSWER]")
ids = tok.encode(prompt)
print(f"prompt tokens: {len(ids)}")

def time_generation(m, n_new=24):
    with torch.no_grad():
        for _ in range(2):
            m(torch.tensor([ids], dtype=torch.long))
        t0 = time.perf_counter()
        m(torch.tensor([ids], dtype=torch.long))
        prefill_ms = (time.perf_counter() - t0) * 1000
        t0 = time.perf_counter()
        out = list(m.generate(list(ids), max_tokens=n_new, temperature=0.0))
        gen_ms = (time.perf_counter() - t0) * 1000
    return prefill_ms, gen_ms, out

m7 = NanoCore(cfg7).eval()
pre_init, gen_init, _ = time_generation(m7)
dezero(m7, 11)
pre_dz, gen_dz, out = time_generation(m7)

print(f"\n{'':28s}{'prefill':>10s}{'24 tokens':>12s}{'ms/token':>11s}")
print(f"{'as shipped (blocks inert)':28s}{pre_init:10.1f}{gen_init:12.1f}{gen_init/24:11.1f}")
print(f"{'de-zeroed (real forward)':28s}{pre_dz:10.1f}{gen_dz:12.1f}{gen_dz/24:11.1f}")
print(f"\nclaimed in ARCHITECTURE.md: 15-35 ms")
print(f"-> the claim is short by ~{gen_dz/35:.0f}x (at 35 ms) to ~{gen_dz/15:.0f}x (at 15 ms)")
print(f"\ndecoded so far: {tok.decode(out)[:120]!r}")

V7 = {"prefill_init_ms": pre_init, "generate_init_ms": gen_init,
      "prefill_dezero_ms": pre_dz, "generate_dezero_ms": gen_dz,
      "ms_per_token_dezero": gen_dz / 24, "claimed_ms": [15, 35]}

## 11. V9 — the cloud entry point

`notebooks/train_colab.ipynb` is the documented way to launch training. Parse it.

In [ ]:
nb_path = Path("notebooks/train_colab.ipynb")
raw = nb_path.read_text()
try:
    json.loads(raw)
    ok, err = True, None
except Exception as e:
    ok, err = False, f"{type(e).__name__}: {e}"

print(f"parseable as JSON: {ok}")
if not ok:
    print(f"  {err}")
    m = __import__("re").search(r"line (\d+)", err)
    if m:
        n = int(m.group(1))
        lines = raw.splitlines()
        for i in range(max(0, n - 3), min(len(lines), n + 1)):
            print(f"  {i+1:4d}| {lines[i]}")

V9 = {"notebook_parses": ok, "error": err}

## 12. V4 — the quality gate

The gate's own summary is `all_passed = all(ok is True or ok == "skipped")`. The committed
`models/quality_gate_report.json` records **Test Suite: skipped** with **all_passed: true**.

Run the real script two ways: with the expensive checks skipped, then in full.

In [ ]:
def run_gate(out_name, *extra, timeout=1500):
    out = f"/kaggle/working/{out_name}"
    cmd = [sys.executable, "scripts/quality_gate.py", "--hardware", "t4_colab",
           "--output", out, *extra]
    t0 = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=str(REPO), timeout=timeout)
    return r, time.time() - t0, json.loads(Path(out).read_text())

print("=" * 70)
print("GATE A: --skip-tests --skip-dry-run  (cheap)")
print("=" * 70)
ra, ta, gate_a = run_gate("gate_a.json", "--skip-tests", "--skip-dry-run")
print(ra.stdout[-2500:])
if ra.stderr.strip():
    print("STDERR:", ra.stderr[-800:])
print(f"exit code: {ra.returncode}   ({ta:.1f}s)")

print("\ncommitted report on record :", json.load(open("models/quality_gate_report.json"))["checks"])
print("fresh report (gate A)      :", gate_a["checks"])
print("gate A all_passed          :", gate_a["all_passed"])
skipped = [c["name"] for c in gate_a["checks"] if c["status"] == "skipped"]
print("skipped checks             :", skipped)
print(f"-> 'Safe to launch training' asserted while {len(skipped)} checks never ran")

V4a = {"exit_code": ra.returncode, "all_passed": gate_a["all_passed"],
       "checks": gate_a["checks"], "skipped": skipped}

In [ ]:
print("=" * 70)
print("GATE B: full run (dry run + test suite)")
print("=" * 70)
try:
    rb, tb, gate_b = run_gate("gate_b.json")
    print(rb.stdout[-5000:])
    if rb.stderr.strip():
        print("STDERR:", rb.stderr[-2000:])
    print(f"exit code: {rb.returncode}   ({tb:.1f}s)")
    print("\nchecks:", gate_b["checks"])
    print("all_passed:", gate_b["all_passed"])
    V4b = {"ran": True, "exit_code": rb.returncode, "all_passed": gate_b["all_passed"],
           "checks": gate_b["checks"], "seconds": tb}
except subprocess.TimeoutExpired as e:
    print(f"gate B timed out after {e.timeout}s")
    V4b = {"ran": False, "error": "timeout"}

# The dry-run checkpoint is ~1GB (fp32 weights + AdamW moments) for a 20-step smoke test.
ckpt = Path("models/nanocore-s1-dryrun.pt")
if ckpt.exists():
    print(f"\ndry-run checkpoint: {ckpt.stat().st_size/1e9:.2f} GB for a 20-step smoke test")
    V4b["checkpoint_gb"] = ckpt.stat().st_size / 1e9

## 13. Package the result

Written to `/kaggle/working` and bundled into a single `artifacts.zip`, because
`kaggle kernels output` fails on output subfolders.

In [ ]:
import shutil

results = {
    "revision_under_test": "mauricew/nanocore-s1-worktree (uploaded working tree)",
    "accelerator": gpu, "torch": torch.__version__,
    "V0_blocks_identity_at_init": V0,
    "V5_parameter_count": V5,
    "V6_window_pattern_inert": V6,
    "V1_decision_accuracy_is_random": V1,
    "V2_ece_does_not_respond": V2,
    "V3_perplexity_unshifted_targets": V3,
    "V8_tokenizer": V8,
    "V7_latency_cpu": V7,
    "V9_notebook_json": V9,
    "V4a_gate_skipped": V4a,
    "V4b_gate_full": V4b,
}
Path("/kaggle/working/audit_results.json").write_text(json.dumps(results, indent=2, default=str))

# Drop the 1GB smoke-test checkpoint before archiving; keep only the evidence files.
ckpt = Path("models/nanocore-s1-dryrun.pt")
if ckpt.exists():
    ckpt.unlink()

names = [n for n in ("audit_results.json", "gate_a.json", "gate_b.json")
         if (Path("/kaggle/working") / n).exists()]
shutil.make_archive("/kaggle/working/artifacts", "zip", "/kaggle/working", *names)
print("artifacts.zip written with:", names)
print(json.dumps(results, indent=2, default=str)[:2500])